In [34]:
import torch
import torch.nn as nn

torch.manual_seed(42)

CHECK FOR THE GPU!

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

DATA PREPARATION!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# dataset
df = pd.read_csv("fmnist_small.csv")

# split train and test
X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

# Scale only the pixels not the labels!
X_train_scaled = X_train/255.0
X_test_scaled = X_test/255.0

# Convert all to tensors
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.long)

'\nX_train_tensor = torch.from_numpy(X_train_scaled.values).float()\nX_test_tensor = torch.from_numpy(X_test_scaled.values).float()\ny_train_tensor = torch.from_numpy(y_train.values).long()\ny_test_tensor = torch.from_numpy(y_test.values).long()\n'

DATA LOADING!

In [37]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X_train_tensor, y_train_tensor):
        self.X_train_tensor = X_train_tensor
        self.y_train_tensor = y_train_tensor
    def __len__(self):
        return len(self.X_train_tensor)
    def __getitem__(self,idx):
        return self.X_train_tensor[idx], self.y_train_tensor[idx]

train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, pin_memory=True)

MODEL ARCHITECTURE!

In [38]:
class MyNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )
  def forward(self, x):
    return self.model(x)

In [39]:
from torch import optim

# set learning rate and epochs
epochs = 10
learning_rate = 0.1

# instatiate the model
model = MyNN(X_train.shape[1])
model = model.to(device)
# loss function
loss_fun = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.SGD(model.parameters(), lr= learning_rate)

TRAINING LOOP!

In [40]:
# training loop
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    # zero the prev grads
    optimizer.zero_grad()
    # move the tensor data to GPU!
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    # forward pass
    outputs = model(batch_features)
    # calculate loss
    loss = loss_fun(outputs, batch_labels)
    # back pass
    optimizer.zero_grad()
    loss.backward()
    # update grads
    optimizer.step()
    # calculate loss for all items in a batch
    total_epoch_loss = total_epoch_loss + loss.item()
  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 1.3354568207263946
Epoch: 2 , Loss: 0.7806735883156458
Epoch: 3 , Loss: 0.6551915238300959
Epoch: 4 , Loss: 0.5828121983011564
Epoch: 5 , Loss: 0.5376128753026327
Epoch: 6 , Loss: 0.49668346107006073
Epoch: 7 , Loss: 0.45596673876047134
Epoch: 8 , Loss: 0.4423156398534775
Epoch: 9 , Loss: 0.42108585422237715
Epoch: 10 , Loss: 0.3986667436361313


EVALUATION LOOP ON TEST DATA!

In [41]:
# set model to eval mode
model.eval()

# evaluation code
total = len(y_test_tensor)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.805


EVALUATION LOOP ON TRAIN DATA!

In [42]:
# set model to eval mode
model.eval()

# evaluation code
total = len(y_train_tensor)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.838125
